# Imports

In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB

from scipy.stats import shapiro, ttest_rel, wilcoxon

# We load the Spambase corpus

In [ ]:
ds = fetch_openml("spambase", version=1, as_frame=True)

X = ds.data.values
y = ds.target.astype(int).values

print("Dataset:", ds.details.get("name", "spambase"))
print("X shape:", X.shape)
print("Class distribution (0/1):", np.bincount(y))

Dataset: spambase
X shape: (4601, 57)
Class distribution (0/1): [2788 1813]


# Two classifiers: Logistic regression and Gaussian Naive Bayes

In [ ]:
#classifier A: Logistic Regression
clf_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=5000, solver="liblinear", random_state=0))
])

#classifier B: Gaussian Naive Bayes
clf_nb = GaussianNB()

print(clf_lr)
print(clf_nb)

Pipeline(steps=[('scaler', StandardScaler()),
                ('lr',
                 LogisticRegression(max_iter=5000, random_state=0,
                                    solver='liblinear'))])
GaussianNB()


# Validation method and metric

In [ ]:
k_folds = 10
metric = "accuracy"

# I will run the experiment 20 times, changing the seed each time:
seeds = list(range(1, 21))

print("Number of runs:", len(seeds))
print("Seeds:", seeds[:5], "...", seeds[-3:])

Number of runs: 20
Seeds: [1, 2, 3, 4, 5] ... [18, 19, 20]


#We run 20 repeated CV evaluations (paired)

In [ ]:
scores_lr = []
scores_nb = []

for seed in seeds:
    cv = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=seed)

    # Cross-validated accuracy (mean over folds) for each model
    acc_lr = cross_val_score(clf_lr, X, y, cv=cv, scoring=metric).mean()
    acc_nb = cross_val_score(clf_nb, X, y, cv=cv, scoring=metric).mean()

    scores_lr.append(acc_lr)
    scores_nb.append(acc_nb)

scores_lr = np.array(scores_lr)
scores_nb = np.array(scores_nb)
diff = scores_lr - scores_nb

results = pd.DataFrame({
    "seed": seeds,
    "acc_LR": scores_lr,
    "acc_NB": scores_nb,
    "diff_LR_minus_NB": diff
})

results

,seed,acc_LR,acc_NB,diff_LR_minus_NB
0,1,0.925021,0.819823,0.105198
1,2,0.926103,0.822213,0.103890
2,3,0.923926,0.822425,0.101501
3,4,0.925015,0.822003,0.103012
4,5,0.925450,0.821560,0.103890
5,6,0.925671,0.821555,0.104115
6,7,0.926104,0.820255,0.105849
7,8,0.925236,0.820478,0.104758
8,9,0.925018,0.821341,0.103677
9,10,0.924583,0.820687,0.103896


# Summary of results

### Now I have 20 accuracy values for each classifier. Since both models were evaluated using the same cross-validation splits in every run (same seed), the comparison is paired. This ensures that each pair of scores comes from identical data partitions.

### The difference in accuracy between the two is consistently around 0.10–0.11, meaning that Logistic Regression is about 10–11 percentage points better than Naive Bayes. Although this gap is stable, I will now perform a statistical test to confirm whether it is statistically significant.

# Normality test (Shapiro–Wilk)

In [ ]:
from scipy.stats import shapiro

W, p_shapiro = shapiro(diff)

print("Shapiro-Wilk statistic:", W)
print("Shapiro-Wilk p-value:", p_shapiro)

Shapiro-Wilk statistic: 0.9587156259829503
Shapiro-Wilk p-value: 0.5185210548830126


### Note: The Shapiro-Wilk p-value (0.5185) is greater than 0.05, so I do not reject normality. This means that the differences between the two models can be considered approximately normally distributed. Therefore, I will use a paired t-test for the statistical comparison.

# Paired t-test

In [ ]:
from scipy.stats import ttest_rel

t_stat, p_value = ttest_rel(scores_lr, scores_nb)

print("Paired t-test statistic:", t_stat)
print("Paired t-test p-value:", p_value)

Paired t-test statistic: 311.88306000572015
Paired t-test p-value: 1.0432762229473477e-36


## Note: As we can observe the paired t-test produced a p-value of 1.04 x 10⁻³⁶, which is much smaller than 0.05. This means the difference in performance between the two models is statistically significant. Since Logistic Regression consistently outperforms Naive Bayes by 10-11 percentaje points as we saw before, Logisitic regression is definitively the best classifier for the Spambase dataset.
